# Process and save info of CS calculated using FLUXCOM-X and inversion models (CarboScope and CAMS)

#### Libraries

In [1]:
from data_compilation import *
from figures_functions import *

## FLUXCOM-X

In [2]:
nee=get_neeFLUXCOM("2001-01","2021-12") # it takes a long time.
nee=nee*(-1) # FLUXCOM-X considers fluxes positive into the atmosphere.

In [3]:
nee["time"]=nee["time"].astype("datetime64[ns]")

#### Common period

In [4]:
d1='2001-01'; d2='2019-12'
nee=nee.sel(time=slice(d1,d2))

In [5]:
Mt=nee.cumulative_integrate('time',datetime_unit='D')/30.4/12 # mass [kg C m-2].
cs=Mt.integrate("time",datetime_unit='D')/30.4/12 # carbon sequestration [kg C m-2 yr].

# Mt=nee.cumulative_integrate("time") # mass [kg C m-2].
# cs=Mt.integrate("time") # carbon sequestration [kg C m-2 yr] # without substracting the initial value.
dy=Mt.indexes['time'][-1].year-Mt.indexes['time'][1].year
cs2=Mt.integrate("time")-(Mt.isel(time=1)*dy) # carbon sequestration [kg C m-2 yr] # substracting the initial value times T.
cs_cumt=Mt.cumulative_integrate("time")
cs_cumxy=cs_cumt.sum(["lon"]).sum(["lat"])
Mt_xy=Mt.sum("lon").sum("lat")

#### Save results

In [6]:
Mtnee=xr.Dataset({'FLUXCOM':Mt})
csnee=xr.Dataset({'FLUXCOM':cs})
# Mean NEE
x_mean=nee.mean("time")
x_mean=xr.Dataset({'FLUXCOM':x_mean})

Mtnee=Mtnee.assign_attrs(units="kgC m-2", description="Carbon mass calculated with NEE from FLUXCOM-X.")
csnee=csnee.assign_attrs(units="kgC m-2 yr", description="Carbon sequestration calculated with NEE from FLUXCOM-X.")
x_mean=x_mean.assign_attrs(units="kgC m-2 yr-1", description="NEE from FLUXCOM-X.")

# # With regridding
Mtnee.to_netcdf('/home/data/ResultsCS/MT_nee_Fluxcom_2001-2019.nc')
csnee.to_netcdf('/home/data/ResultsCS/CS_nee_Fluxcom_2001-2019.nc')
x_mean.to_netcdf('/home/data/ResultsCS/nee_mean_Fluxcom_2001-2019.nc')
# without regridding.\n",
# Mtnee_all.to_netcdf('/home/data/ResultsCS/MT_nee_Fluxcom_oscale_v2.nc'),
# csnee_all.to_netcdf('/home/data/ResultsCS/CS_nee_Fluxcom_oscale_v2.nc')

# Mtnee.to_netcdf('/home/data/ResultsCS/MT_nee_Fluxcom_2001-2014.nc')
#csnee.to_netcdf('/home/data/ResultsCS/CS_nee_Fluxcom_2001-2014.nc')

#### Save data of nee for latitudinal zones accumulations

In [7]:
titles=['Global','Tropics','Mid_latitudes','High_latitudes']
var=nee

for k in range(len(titles)):
    gc.collect()
    #for k in range(1,2):
    if k==0:varx=var # Global
    if k==1:mask=((var.lat <= 22.5)&(var.lat >= -22.5)) # Tropics
    if k==2:
        mask1=(var.lat > 22.5)&(var.lat < 55.0) # mid latitudes.
        mask2=(var.lat < -22.5) # mid latitudes.
    if k==3:mask=(var.lat >= 55.0) # high latitudes
    if k!=0: varx=var.where(mask,drop=True)

    if k==2:
        varx1=var.where(mask1,drop=True);varx2=var.where(mask2,drop=True)
        varxy=calc_spatial_integral(varx1)+calc_spatial_integral(varx2) 
    else:varxy=calc_spatial_integral(varx)
    varxy.to_netcdf('/home/data/ResultsCS/variables_latzones/FLUXCOM_nee_'+titles[k]+'_2001-2019.nc')

In [8]:
Mtnee

<xarray.Dataset> Size: 118MB
Dimensions:  (lat: 180, lon: 360, time: 228)
Coordinates:
  * lat      (lat) float32 720B -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon      (lon) float32 1kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * time     (time) datetime64[ns] 2kB 2001-01-01 2001-02-01 ... 2019-12-01
Data variables:
    FLUXCOM  (time, lat, lon) float64 118MB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    units:        kgC m-2
    description:  Carbon mass calculated with NEE from FLUXCOM-X.

## CarboScope

### NEE

In [8]:
nee=get_neeCarboscope("2001-01","2021-12")
nee=nee*(-1) # CarboScope considers fluxes positive into the atmosphere.
nee["time"]=nee["time"].astype("datetime64[ns]")

/home/estefania/anaconda3/envs/xesmf_env/lib/python3.12/site-packages/xarray/core/computation.py:320: PerformanceWarning: Regridding is increasing the number of chunks by a factor of 8.0, you might want to specify sizes in `output_chunks` in the regridder call. Default behaviour is to preserve the chunk sizes from the input (90, 90).
  result_var = func(*data_vars)


In [9]:
d1='2001-01'; d2='2019-12'
nee=nee.sel(time=slice(d1,d2))

In [10]:
Mt=nee.cumulative_integrate("time",datetime_unit='D')/30.4/12 # mass [kg C m-2].
cs=Mt.integrate("time",datetime_unit='D')/30.4/12 # carbon sequestration [kg C m-2 yr]."

Mtnee=xr.Dataset({'CarboScope':Mt})
csnee=xr.Dataset({'CarboScope':cs})

# Mean NEE
x_mean=nee.mean("time")
x_mean=xr.Dataset({'nee':x_mean})

Mtnee=Mtnee.assign_attrs(units="kgC m-2", description="Carbon mass calculated with NEE (NBP+Fire) from CarboScope.")
csnee=csnee.assign_attrs(units="kgC m-2 yr", description="Carbon sequestration calculated with NEE (NBP+Fire) from CarboScope.")
x_mean=x_mean.assign_attrs(units="kgC m-2 yr-1", description="NEE from CarboScope (NBP+Fire).")

# Mtnee.to_netcdf('/home/data/ResultsCS/MT_nee_CarboScope.nc')\n",
# csnee.to_netcdf('/home/data/ResultsCS/CS_nee_CarboScope.nc')\n",
Mtnee.to_netcdf('/home/data/ResultsCS/MT_nee_CarboScope_2001-2019.nc')
csnee.to_netcdf('/home/data/ResultsCS/CS_nee_CarboScope_2001-2019.nc')

x_mean.to_netcdf('/home/data/ResultsCS/nee_mean_CarboScope_2001-2019.nc')


### NBP

In [11]:
nbp=get_nbpCarboscope("1987-01","2023-12") 
nbp=nbp*(-1) # CarboScope considers fluxes positive into the atmosphere.
nbp["time"]=nbp["time"].astype("datetime64[ns]")

In [12]:
d1='2001-01'; d2='2019-12'
nbp=nbp.sel(time=slice(d1,d2))

In [13]:
Mt=nbp.cumulative_integrate("time",datetime_unit='D')/30.4/12 # mass [kg C m-2].
cs=Mt.integrate("time",datetime_unit='D')/30.4/12 # carbon sequestration [kg C m-2 yr]."


Mtnbp=xr.Dataset({'CarboScope':Mt})
csnbp=xr.Dataset({'CarboScope':cs})

# Mean NBP
x_mean=nbp.mean("time")
x_mean=xr.Dataset({'CarboSCope':x_mean})

Mtnbp=Mtnbp.assign_attrs(units="kgC m-2", description="Carbon mass calculated with NBP from CarboScope.")
csnbp=csnbp.assign_attrs(units="kgC m-2 yr", description="Carbon sequestration calculated with NBP from CarboScope.")
x_mean=x_mean.assign_attrs(units="kgC m-2 yr-1", description="NBP from CarboScope.")

# Mtnbp.to_netcdf('/home/data/ResultsCS/MT_nbp_CarboScope.nc')
# csnbp.to_netcdf('/home/data/ResultsCS/CS_nbp_CarboScope.nc')
# x_mean.to_netcdf('/home/data/ResultsCS/nbp_mean_CarboScope.nc')

Mtnbp.to_netcdf('/home/data/ResultsCS/MT_nbp_CarboScope_2001-2019.nc')
csnbp.to_netcdf('/home/data/ResultsCS/CS_nbp_CarboScope_2001-2019.nc')
x_mean.to_netcdf('/home/data/ResultsCS/nbp_mean_CarboScope_2001-2019.nc')


In [23]:
Mtnbp

<xarray.Dataset> Size: 116MB
Dimensions:     (lat: 180, lon: 360, time: 224)
Coordinates:
  * lat         (lat) float32 720B -89.5 -88.5 -87.5 -86.5 ... 87.5 88.5 89.5
  * lon         (lon) float32 1kB 0.5 1.5 2.5 3.5 ... 356.5 357.5 358.5 359.5
  * time        (time) datetime64[ns] 2kB 2001-01-01 2001-02-01 ... 2019-12-01
Data variables:
    CarboScope  (time, lat, lon) float64 116MB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    units:        kgC m-2
    description:  Carbon mass calculated with NBP from CarboScope.

#### Save data of nbp for latitudinal zones accumulations¶

In [14]:
titles=['Global','Tropics','Mid_latitudes','High_latitudes']
var=nbp

for k in range(len(titles)):
    gc.collect()
    #for k in range(1,2):
    if k==0:varx=var # Global
    if k==1:mask=((var.lat <= 22.5)&(var.lat >= -22.5)) # Tropics
    if k==2:
        mask1=(var.lat > 22.5)&(var.lat < 55.0) # mid latitudes.
        mask2=(var.lat < -22.5) # mid latitudes.
    if k==3:mask=(var.lat >= 55.0) # high latitudes
    if k!=0: varx=var.where(mask,drop=True)

    if k==2:
        varx1=var.where(mask1,drop=True);varx2=var.where(mask2,drop=True)
        varxy=calc_spatial_integral(varx1)+calc_spatial_integral(varx2) 
    else:varxy=calc_spatial_integral(varx)
    varxy.to_netcdf('/home/data/ResultsCS/variables_latzones/CarboScope_nbp_'+titles[k]+'_2001-2019.nc')

## CAMS

In [15]:
nbp=get_nbpCAMS("1979-01","2020-12") 
nbp=nbp*(-1) # CAMS considers fluxes positive into the atmosphere.\n",
nbp["time"]=nbp["time"].astype("datetime64[ns]")


In [16]:
d1='2001-01'; d2='2019-12'
nbp=nbp.sel(time=slice(d1,d2))

In [17]:
Mt=nbp.cumulative_integrate("time",datetime_unit='D')/30.4/12 # mass [kg C m-2].
cs=Mt.integrate("time",datetime_unit='D')/30.4/12 # carbon sequestration [kg C m-2 yr]."

Mtnbp=xr.Dataset({'CAMS':Mt})
csnbp=xr.Dataset({'CAMS':cs})

# Mean NBP
x_mean=nbp.mean("time")
x_mean=xr.Dataset({'CAMS':x_mean})

Mtnbp=Mtnbp.assign_attrs(units="kgC m-2", description="Carbon mass calculated with NBP from CAMS.")
csnbp=csnbp.assign_attrs(units="kgC m-2 yr", description="Carbon sequestration calculated with NBP from CAMS.")
x_mean=x_mean.assign_attrs(units="kgC m-2 yr-1", description="NBP from CAMS.")

# Mtnbp.to_netcdf('/home/data/ResultsCS/MT_nbp_CAMS.nc')
# csnbp.to_netcdf('/home/data/ResultsCS/CS_nbp_CAMS.nc')
# x_mean.to_netcdf('/home/data/ResultsCS/nbp_mean_CAMS.nc')

Mtnbp.to_netcdf('/home/data/ResultsCS/MT_nbp_CAMS_2001-2019.nc')
csnbp.to_netcdf('/home/data/ResultsCS/CS_nbp_CAMS_2001-2019.nc')
x_mean.to_netcdf('/home/data/ResultsCS/nbp_mean_CAMS_2001-2019.nc')

#### Save data of nbp for latitudinal zones accumulations¶

In [18]:
titles=['Global','Tropics','Mid_latitudes','High_latitudes']
var=nbp

for k in range(len(titles)):
    gc.collect()
    #for k in range(1,2):
    if k==0:varx=var # Global
    if k==1:mask=((var.lat <= 22.5)&(var.lat >= -22.5)) # Tropics
    if k==2:
        mask1=(var.lat > 22.5)&(var.lat < 55.0) # mid latitudes.
        mask2=(var.lat < -22.5) # mid latitudes.
    if k==3:mask=(var.lat >= 55.0) # high latitudes
    if k!=0: varx=var.where(mask,drop=True)

    if k==2:
        varx1=var.where(mask1,drop=True);varx2=var.where(mask2,drop=True)
        varxy=calc_spatial_integral(varx1)+calc_spatial_integral(varx2) 
    else:varxy=calc_spatial_integral(varx)
    varxy.to_netcdf('/home/data/ResultsCS/variables_latzones/CAMS_nbp_'+titles[k]+'_2001-2019.nc')

In [29]:
Mtnbp

<xarray.Dataset> Size: 116MB
Dimensions:  (lat: 180, lon: 360, time: 224)
Coordinates:
  * lat      (lat) float32 720B -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon      (lon) float32 1kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * time     (time) datetime64[ns] 2kB 2001-01-01 2001-02-01 ... 2019-12-01
Data variables:
    CAMS     (time, lat, lon) float64 116MB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    units:        kgC m-2
    description:  Carbon mass calculated with NBP from CAMS.